<a href="https://colab.research.google.com/github/jonathanastudillo-lab/Ejercicio/blob/main/House_Prices_Advanced_Regression_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

primero se cargan las librerías

SSL para cargar desde internet un archivo
pandas para empezar a trabajar en la base de datos

In [11]:
import ssl
import pandas as pd
import numpy as np


ssl._create_default_https_context = ssl._create_unverified_context
# Usamos la URL raw para obtener solo el contenido del CSV
url = "https://raw.githubusercontent.com/jonathanastudillo-lab/Ejercicio/main/train_houseprice.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


la base tiene 1460 filas y 81 columnas

2) tratamiento de datos

2.1 estadística descriptiva

In [12]:
# 1) Ver tamaño y primeras filas
print("Dimensión del DataFrame:", df.shape)
display(df.head())

# 2) Separar variables numéricas y categóricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Cantidad de variables numéricas:", len(num_cols))
print("Cantidad de variables categóricas:", len(cat_cols))

print("\nVariables numéricas:")
print(num_cols)

print("\nVariables categóricas:")
print(cat_cols)

# 3) Resumen estadístico de variables numéricas
resumen_numerico = df[num_cols].describe().T
display(resumen_numerico)

# 4) Función para winsorizar entre percentil 1 y percentil 99
def winsorize_p1_p99(df, cols):
    df_out = df.copy()
    bounds = {}

    for c in cols:
        p1 = df_out[c].quantile(0.01)
        p99 = df_out[c].quantile(0.99)
        bounds[c] = (p1, p99)
        df_out[c] = df_out[c].clip(lower=p1, upper=p99)

    return df_out, bounds

Dimensión del DataFrame: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Cantidad de variables numéricas: 38
Cantidad de variables categóricas: 43

Variables numéricas:
['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice']

Variables categóricas:
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 

,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


en la varaibles categóricas no tiene sentido hacer análisis:

'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition'

In [14]:
# 5) Aplicar winsorización solo a variables numéricas
df_w, bounds = winsorize_p1_p99(df, num_cols)

# 6) Convertir bounds a tabla para verlo ordenado
bounds_df = pd.DataFrame(bounds, index=["p1", "p99"]).T
bounds_df.index.name = "Variable"

print("\nPercentiles usados para winsorización:")
display(bounds_df)

# 7) Comparación simple antes y después
print("Resumen ANTES de winsorizar:")
display(df[num_cols].describe().T)

print("Resumen DESPUÉS de winsorizar:")
display(df_w[num_cols].describe().T)


Percentiles usados para winsorización:


,p1,p99
Variable,,
Id,15.59,1445.41
MSSubClass,20.00,190.00
LotFrontage,21.00,141.00
LotArea,1680.00,37567.64
OverallQual,3.00,10.00
OverallCond,3.00,9.00
YearBuilt,1899.18,2009.00
YearRemodAdd,1950.00,2009.00
MasVnrArea,0.00,791.92


Resumen ANTES de winsorizar:


,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


Resumen DESPUÉS de winsorizar:


,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.343430,15.59,365.75,730.5,1095.25,1445.41
MSSubClass,1460.0,56.897260,42.300571,20.00,20.00,50.0,70.00,190.00
LotFrontage,1201.0,69.600333,21.723531,21.00,59.00,69.0,80.00,141.00
LotArea,1460.0,10063.006575,5062.299090,1680.00,7553.50,9478.5,11601.50,37567.64
OverallQual,1460.0,6.104110,1.369444,3.00,5.00,6.0,7.00,10.00
OverallCond,1460.0,5.580137,1.098826,3.00,5.00,5.0,6.00,9.00
YearBuilt,1460.0,1971.413493,29.806332,1899.18,1954.00,1973.0,2000.00,2009.00
YearRemodAdd,1460.0,1984.861644,20.640499,1950.00,1967.00,1994.0,2004.00,2009.00
MasVnrArea,1452.0,101.174793,168.046772,0.00,0.00,0.0,166.00,791.92
BsmtFinSF1,1460.0,438.530240,429.049203,0.00,0.00,383.5,712.25,1572.41


Se identificaron las variables numéricas y categóricas de la base House Prices.

 Luego, para reducir el efecto de valores extremos, se aplicó winsorización a las variables numéricas, reemplazando los valores inferiores al percentil 1 y superiores al percentil 99 por esos límites.

Ahora veremos una variable puntual "SALES"

para ver como quedó antes y después

In [19]:
print("variable Sale")
bounds_df.loc["SalePrice"]

variable Sale


,SalePrice
p1,61815.97
p99,442567.01


In [18]:


print("Antes:")
print(df["SalePrice"].describe())

print("\nDespués:")
print(df_w["SalePrice"].describe())

Antes:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Después:
count      1460.000000
mean     179926.416233
std       74052.658892
min       61815.970000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      442567.010000
Name: SalePrice, dtype: float64


Ahora revisamos y tratamos los missing value

In [21]:
# Conteo de missing
missing_count = df_w.isna().sum().sort_values(ascending=False)
missing_pct = (df_w.isna().mean() * 100).sort_values(ascending=False)

pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct}).head(20)

,missing_count,missing_pct
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageQual,81,5.547945
GarageFinish,81,5.547945
GarageType,81,5.547945


las variables PoolQC MiscFeature Alley Fence MasVnrType FireplaceQu se considerarán variables que tienen mas del 20% de sus valores como missing value

lo interpretaremos como que varias de esas variables faltan porque la característica no existe en esa casa, por ejemplo:

PoolQC → si no tiene piscina

Alley → si no tiene callejón de acceso

Fence → si no tiene cerco

FireplaceQu → si no tiene chimenea

GarageQual, GarageType, etc. → si no tiene garaje

BsmtExposure, BsmtCond, etc. → si no tiene sótano

O sea, aquí el missing muchas veces significa **No existe** y no simplemente **faltó registrar**

In [22]:
import pandas as pd
import numpy as np

# Copia de trabajo
df_clean = df_w.copy()

# 1) Eliminar columnas con exceso extremo de missing
drop_cols = ["PoolQC", "MiscFeature", "Alley", "Fence"]
drop_cols = [c for c in drop_cols if c in df_clean.columns]
df_clean.drop(columns=drop_cols, inplace=True)

# 2) Categóricas donde el NA significa "no existe esa característica"
fill_none_cols = [
    "MasVnrType",
    "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]

for c in fill_none_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna("None")

# 3) Numéricas donde el NA también representa ausencia real
fill_zero_cols = [
    "MasVnrArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath",
    "GarageCars", "GarageArea", "GarageYrBlt"
]

for c in fill_zero_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna(0)

# 4) LotFrontage: imputar con la mediana por Neighborhood
if "LotFrontage" in df_clean.columns and "Neighborhood" in df_clean.columns:
    df_clean["LotFrontage"] = df_clean.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )
    # por si algún barrio quedara completo en NA
    df_clean["LotFrontage"] = df_clean["LotFrontage"].fillna(df_clean["LotFrontage"].median())

# 5) Variables categóricas restantes: imputar con la moda
cat_cols_clean = df_clean.select_dtypes(exclude=[np.number]).columns.tolist()
for c in cat_cols_clean:
    if df_clean[c].isna().sum() > 0:
        df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])

# 6) Variables numéricas restantes: imputar con la mediana
num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_clean:
    if df_clean[c].isna().sum() > 0:
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())

# 7) Verificación final
missing_final = df_clean.isna().sum().sort_values(ascending=False)
print(missing_final.head(20))
print("\nTotal de missing restantes:", df_clean.isna().sum().sum())
print("\nDimensión final:", df_clean.shape)

Id              0
MSSubClass      0
MSZoning        0
LotFrontage     0
LotArea         0
Street          0
LotShape        0
LandContour     0
Utilities       0
LotConfig       0
LandSlope       0
Neighborhood    0
Condition1      0
Condition2      0
BldgType        0
HouseStyle      0
OverallQual     0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
dtype: int64

Total de missing restantes: 0

Dimensión final: (1460, 77)


En la base House Prices, los valores faltantes se trataron de forma diferenciada según su significado. Se eliminaron variables con exceso extremo de missing, se imputó “None” en variables categóricas donde la ausencia representa que la vivienda no posee dicha característica, se imputó 0 en variables numéricas asociadas a ausencia real, y en el resto se utilizaron medidas robustas como la mediana o la moda.

--------------
ahora veremos si existe alguna correlación entrevariables, porque  en muchos modelos, sobre todo los lineales, meter dos variables muy correlacionadas puede:

meter ruido innecesario

dificultar la interpretación

generar multicolinealidad

hacer el modelo menos limpio

Ejemplo simple:

GarageCars = cantidad de autos en garaje

GarageArea = tamaño del garaje

Probablemente estén bastante correlacionadas, porque a más autos, normalmente más área

Entonces a veces conviene dejar solo una

In [ ]:
# Recalcular columnas numéricas desde df_clean
num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Matriz de correlación
corr = df_clean[num_cols_clean].corr(method="pearson")

# Ver matriz completa
display(corr)

# Ver correlaciones más fuertes entre variables (sin repetir pares)
corr_abs = corr.abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

top_pairs = upper.stack().sort_values(ascending=False)
display(top_pairs.head(20))